# TP1 – Sistema de IA para o Jogo da Velha

**Objetivo:** Classificar o estado de um tabuleiro 3×3 em uma das cinco classes:
- **Tem jogo** – partida em andamento, sem ameaça imediata de vitória
- **Possibilidade de Fim de Jogo** – um jogador pode vencer na próxima jogada
- **Empate** – tabuleiro cheio sem vencedor
- **O vence** – O possui três em linha
- **X vence** – X possui três em linha

**Dataset base:** UCI Tic-Tac-Toe Endgame  
https://archive.ics.uci.edu/dataset/101/tic+tac+toe+endgame
---

> **Papel deste notebook no projeto**
>
> Este notebook cobre os itens **2 (dataset)** e **4 (divisão do dataset)** do enunciado:
> ele parte do arquivo bruto do UCI, diagnostica seus problemas, reclassifica e gera
> estados sintéticos, e grava a divisão física em `data/processed/`.
>
> As seções 10 a 12 (árvore de decisão e k-NN) ficam aqui como **registro exploratório**,
> com os gráficos usados no relatório. Os experimentos **oficiais** de todos os cinco
> algoritmos ficam em `models/`, onde seguem um protocolo único e comparável
> (ver `ttt/experiment.py`). Para reproduzir os números do relatório, rode:
>
> ```bash
> python models/comparar.py
> ```


## 1. Importações e Configurações

In [ ]:
import random
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score
)

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

COLS = ['tl', 'tm', 'tr', 'ml', 'mm', 'mr', 'bl', 'bm', 'br']
CLASSES = ['Tem jogo', 'Possibilidade de Fim de Jogo', 'Empate', 'O vence', 'X vence']

print('Bibliotecas carregadas com sucesso.')

## 2. Carregamento do Dataset UCI

In [ ]:
uci_raw = pd.read_csv(
    '../data/raw/tic-tac-toe.data',
    header=None,
    names=COLS + ['target_uci']
)

print('Shape:', uci_raw.shape)
print('\nDistribuição das classes originais:')
print(uci_raw['target_uci'].value_counts())
print('\nAmostra:')
uci_raw.head()

## 3. Análise e Diagnóstico do Dataset

### Problemas identificados

| # | Problema | Impacto |
|---|---|---|
| 1 | **Apenas 2 classes** (`positive`/`negative`) | Precisamos de 5 classes |
| 2 | **Somente estados finais de jogo** | Faltam "Tem jogo" e "Possibilidade de Fim de Jogo" |
| 3 | **Dataset desbalanceado** (626 vs 332) | Pode enviesar modelos |

### Plano de adequação
1. Reclassificar `positive` → `X vence`
2. Reclassificar `negative` → `O vence` ou `Empate` (via inspeção lógica do tabuleiro)
3. Gerar estados sintéticos válidos para as classes `Tem jogo` e `Possibilidade de Fim de Jogo`
4. Balancear para ~200 amostras por classe

In [ ]:
print('Valores nulos:', uci_raw.isnull().sum().sum())
print('\nValores únicos por posição:')
for c in COLS:
    print(f'  {c}: {sorted(uci_raw[c].unique())}')

## 4. Funções de Análise Lógica do Tabuleiro

In [ ]:
LINHAS_VENCEDORAS = [
    [0,1,2], [3,4,5], [6,7,8],   # linhas
    [0,3,6], [1,4,7], [2,5,8],   # colunas
    [0,4,8], [2,4,6]              # diagonais
]

def vencedor(board):
    for linha in LINHAS_VENCEDORAS:
        vals = [board[i] for i in linha]
        if vals[0] == vals[1] == vals[2] and vals[0] != 'b':
            return vals[0]
    return None

def pode_vencer_prox(board, jogador):
    for linha in LINHAS_VENCEDORAS:
        vals = [board[i] for i in linha]
        if vals.count(jogador) == 2 and vals.count('b') == 1:
            return True
    return False

def classificar(board):
    w = vencedor(board)
    if w == 'x':
        return 'X vence'
    if w == 'o':
        return 'O vence'
    if board.count('b') == 0:
        return 'Empate'
    if pode_vencer_prox(board, 'x') or pode_vencer_prox(board, 'o'):
        return 'Possibilidade de Fim de Jogo'
    return 'Tem jogo'

def tabuleiro_valido(board):
    cx, co = board.count('x'), board.count('o')
    if cx not in (co, co + 1):
        return False
    wx = (vencedor(board) == 'x')
    wo = (vencedor(board) == 'o')
    if wx and wo:
        return False
    if wx and cx != co + 1:
        return False
    if wo and cx != co:
        return False
    return True

# Testes de sanidade
casos = [
    (['x','x','x','o','o','b','b','b','b'], 'X vence'),
    (['o','o','o','x','x','b','x','b','b'], 'O vence'),
    (['x','o','x','x','o','o','o','x','x'], 'Empate'),
    (['x','b','b','b','o','b','b','b','b'], 'Tem jogo'),
    (['x','x','b','o','o','b','b','b','b'], 'Possibilidade de Fim de Jogo'),
]
for b, label in casos:
    r = classificar(b)
    print(f'{label}: {"OK" if r == label else f"ERRO (obtido: {r})"}')

## 5. Reclassificação do Dataset UCI

In [ ]:
def linha_para_board(row):
    return [row[c] for c in COLS]

uci_raw['classe'] = uci_raw.apply(lambda row: classificar(linha_para_board(row)), axis=1)

print('Distribuição após reclassificação:')
print(uci_raw['classe'].value_counts())

cross = pd.crosstab(uci_raw['target_uci'], uci_raw['classe'])
print('\nCruzamento UCI × nova classe:')
print(cross)

### Observações sobre a reclassificação

- **`positive` → `X vence`**: confirmado integralmente (626 instâncias).
- **`negative` → `O vence` + `Empate`**: a inspeção lógica separou os 332 casos corretamente.
- As classes `Tem jogo` e `Possibilidade de Fim de Jogo` **não existem** no dataset original — ele contém apenas estados finais.

Precisamos **gerar** essas instâncias de forma sintética, garantindo validade lógica do tabuleiro.

## 6. Geração de Estados Sintéticos para Classes Ausentes

Estratégia: simular partidas aleatórias e capturar snapshots intermediários do tabuleiro.

In [ ]:
def simular_partida():
    board = ['b'] * 9
    jogadores = itertools.cycle(['x', 'o'])
    estados = []
    for jogador in jogadores:
        vazias = [i for i, v in enumerate(board) if v == 'b']
        if not vazias:
            break
        board[random.choice(vazias)] = jogador
        snapshot = board[:]
        classe = classificar(snapshot)
        estados.append((snapshot, classe))
        if classe in ('X vence', 'O vence', 'Empate'):
            break
    return estados

def gerar_estados(classes_alvo, n_por_classe=500, max_simulacoes=100_000):
    coletados = {c: [] for c in classes_alvo}
    vistos = set()
    for _ in range(max_simulacoes):
        if all(len(v) >= n_por_classe for v in coletados.values()):
            break
        for snapshot, classe in simular_partida():
            if classe in classes_alvo and len(coletados[classe]) < n_por_classe:
                chave = tuple(snapshot)
                if chave not in vistos:
                    vistos.add(chave)
                    coletados[classe].append(snapshot)
    rows = []
    for classe, boards in coletados.items():
        for b in boards:
            row = dict(zip(COLS, b))
            row['classe'] = classe
            rows.append(row)
    return pd.DataFrame(rows)

df_sintetico = gerar_estados(['Tem jogo', 'Possibilidade de Fim de Jogo'], n_por_classe=500)
print('Estados gerados:')
print(df_sintetico['classe'].value_counts())

## 7. Construção do Dataset Balanceado (~200 amostras por classe)

In [ ]:
N_POR_CLASSE = 200

uci_limpo = uci_raw[COLS + ['classe']].copy()
df_todo = pd.concat([uci_limpo, df_sintetico], ignore_index=True)

antes = len(df_todo)
df_todo = df_todo.drop_duplicates(subset=COLS)
print(f'Duplicatas removidas: {antes - len(df_todo)}')

partes = []
for cls in CLASSES:
    subset = df_todo[df_todo['classe'] == cls]
    n = min(N_POR_CLASSE, len(subset))
    partes.append(subset.sample(n=n, random_state=42))
    print(f'{cls}: {len(subset)} disponíveis → {n} usadas')

df = pd.concat(partes, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nDataset final: {df.shape}')
print(df['classe'].value_counts())

In [ ]:
counts = df['classe'].value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASSES, counts.values, color=sns.color_palette('Set2', len(CLASSES)))
ax.set_title('Distribuição das Classes no Dataset Final')
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, str(v), ha='center')
plt.tight_layout()
plt.savefig('../reports/figures/nb01_distribuicao_classes.png', dpi=120)
plt.show()

## 8. Codificação das Features

Convertemos os valores categóricos `{x, o, b}` para numéricos `{2, 1, 0}`:  
X = 2, O = 1, vazio = 0. Essa codificação preserva a semântica de "X joga primeiro".

In [ ]:
ENCODE_CASA = {'b': 0, 'o': 1, 'x': 2}

for c in COLS:
    df[c] = df[c].map(ENCODE_CASA)

le = LabelEncoder()
df['classe_id'] = le.fit_transform(df['classe'])

print('Mapeamento de classes (LabelEncoder):')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

X = df[COLS].values
y = df['classe_id'].values
print(f'\nX shape: {X.shape} | y shape: {y.shape}')

## 9. Divisão do Dataset em Treino e Teste

Estratégia: **70% treino / 30% teste**, com **validação cruzada estratificada (5-fold)**  
aplicada exclusivamente ao conjunto de treino para ajuste de hiperparâmetros.  
O conjunto de teste permanece intocado até a avaliação final, garantindo imparcialidade.

In [ ]:
# Divisão física: 60% treino | 20% validação | 20% teste
# Passo 1 – separa teste (20%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# Passo 2 – separa validação do restante (25% de 80% = 20% do total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f'Treino     : {len(X_train):>4} instâncias  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Validação  : {len(X_val):>4} instâncias  ({len(X_val)/len(X)*100:.0f}%)')
print(f'Teste      : {len(X_test):>4} instâncias  ({len(X_test)/len(X)*100:.0f}%)')

print('\nDistribuição por conjunto:')
header = f'{"Classe":<35} {"Treino":>7} {"Val":>5} {"Teste":>6}'
print(header)
print('-' * len(header))
for i, cls in enumerate(le.classes_):
    print(f'{cls:<35} {(y_train==i).sum():>7} {(y_val==i).sum():>5} {(y_test==i).sum():>6}')

# Salvar os três conjuntos (reprodutibilidade entre algoritmos)
pd.DataFrame(X_train,    columns=COLS).assign(classe_id=y_train).to_csv('../data/processed/treino.csv',    index=False)
pd.DataFrame(X_val,      columns=COLS).assign(classe_id=y_val  ).to_csv('../data/processed/validacao.csv', index=False)
pd.DataFrame(X_test,     columns=COLS).assign(classe_id=y_test ).to_csv('../data/processed/teste.csv',     index=False)
print('\nArquivos treino.csv, validacao.csv e teste.csv salvos.')

# PredefinedSplit: -1 = treino, 0 = validação  (usado no GridSearchCV)
from sklearn.model_selection import PredefinedSplit
import numpy as np

X_tv = np.vstack([X_train, X_val])
y_tv = np.concatenate([y_train, y_val])
split_idx = np.array([-1]*len(X_train) + [0]*len(X_val))
ps = PredefinedSplit(split_idx)


## 10. Algoritmo 1 – Árvore de Decisão

Usamos `GridSearchCV` com **PredefinedSplit** — o conjunto de validação físico é usado
para pontuar cada combinação de hiperparâmetros. O modelo final é retreinado em
treino+validação antes da avaliação no teste.

In [ ]:
param_grid_dt = {
    'max_depth'       : [3, 5, 7, 10, None],
    'criterion'       : ['gini', 'entropy'],
    'min_samples_leaf': [1, 3, 5],
}

gs_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=ps,           # <-- conjunto de validação físico
    scoring='accuracy',
    refit=True,      # retreina em train+val com os melhores params
    n_jobs=-1
)
gs_dt.fit(X_tv, y_tv)

print('Melhores hiperparâmetros (Árvore de Decisão):')
print(gs_dt.best_params_)
print(f'Acurácia no conjunto de VALIDAÇÃO: {gs_dt.best_score_:.4f}')


In [ ]:
resultados_dt = pd.DataFrame(gs_dt.cv_results_)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for criterion, ax in zip(['gini', 'entropy'], axes):
    sub = resultados_dt[
        (resultados_dt['param_criterion'] == criterion) &
        (resultados_dt['param_min_samples_leaf'] == 1)
    ].copy()
    sub['_order'] = sub['param_max_depth'].apply(lambda v: 999 if v is None else int(v))
    sub['_label'] = sub['param_max_depth'].apply(lambda v: 'None' if v is None else str(v))
    sub = sub.sort_values('_order')
    xs = list(range(len(sub)))
    ax.plot(xs, sub['mean_test_score'].tolist(), marker='o', label=criterion)
    ax.fill_between(xs,
        (sub['mean_test_score'] - sub['std_test_score']).tolist(),
        (sub['mean_test_score'] + sub['std_test_score']).tolist(),
        alpha=0.2)
    ax.set_xticks(xs)
    ax.set_xticklabels(sub['_label'].tolist())
    ax.set_title(f'Criterion: {criterion}')
    ax.set_xlabel('max_depth')
    ax.set_ylabel('Acurácia (validação)')
    ax.legend()

plt.suptitle('Árvore de Decisão – Acurácia no Conjunto de Validação')
plt.tight_layout()
plt.savefig('../reports/figures/nb01_dt_curva_validacao.png', dpi=120)
plt.show()


In [ ]:
melhor_dt = gs_dt.best_estimator_   # retreinado em treino+validação
y_pred_dt = melhor_dt.predict(X_test)

acc_dt_val  = gs_dt.best_score_
acc_dt_test = accuracy_score(y_test, y_pred_dt)
print(f'Acurácia VALIDAÇÃO (Árvore de Decisão): {acc_dt_val:.4f}')
print(f'Acurácia TESTE     (Árvore de Decisão): {acc_dt_test:.4f}\n')
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dt,
    display_labels=le.classes_,
    xticks_rotation='vertical',
    ax=ax, colorbar=False
)
ax.set_title('Matriz de Confusão – Árvore de Decisão (Teste)')
plt.tight_layout()
plt.savefig('../reports/figures/nb01_dt_matriz_confusao.png', dpi=120)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(melhor_dt, feature_names=COLS, class_names=le.classes_,
          filled=True, max_depth=3, ax=ax, fontsize=8)
ax.set_title('Árvore de Decisão (primeiros 3 níveis)')
plt.tight_layout()
plt.savefig('../reports/figures/nb01_dt_arvore.png', dpi=120)
plt.show()


## 11. Algoritmo 2 – K-Nearest Neighbors (KNN)

Mesmo protocolo: `GridSearchCV` com `PredefinedSplit` usando o conjunto de validação físico.

In [ ]:
param_grid_knn = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 15],
    'metric'     : ['euclidean', 'manhattan'],
    'weights'    : ['uniform', 'distance'],
}

gs_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=ps,
    scoring='accuracy',
    refit=True,
    n_jobs=-1
)
gs_knn.fit(X_tv, y_tv)

print('Melhores hiperparâmetros (KNN):')
print(gs_knn.best_params_)
print(f'Acurácia no conjunto de VALIDAÇÃO: {gs_knn.best_score_:.4f}')


In [ ]:
resultados_knn = pd.DataFrame(gs_knn.cv_results_)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for metric, ax in zip(['euclidean', 'manhattan'], axes):
    for w in ['uniform', 'distance']:
        sub = resultados_knn[
            (resultados_knn['param_metric'] == metric) &
            (resultados_knn['param_weights'] == w)
        ].sort_values('param_n_neighbors')
        ax.plot(sub['param_n_neighbors'].tolist(),
                sub['mean_test_score'].tolist(), marker='o', label=w)
    ax.set_title(f'Métrica: {metric}')
    ax.set_xlabel('k (n_neighbors)')
    ax.set_ylabel('Acurácia (validação)')
    ax.legend()

plt.suptitle('KNN – Acurácia no Conjunto de Validação')
plt.tight_layout()
plt.savefig('../reports/figures/nb01_knn_curva_validacao.png', dpi=120)
plt.show()


In [ ]:
melhor_knn = gs_knn.best_estimator_
y_pred_knn = melhor_knn.predict(X_test)

acc_knn_val  = gs_knn.best_score_
acc_knn_test = accuracy_score(y_test, y_pred_knn)
print(f'Acurácia VALIDAÇÃO (KNN): {acc_knn_val:.4f}')
print(f'Acurácia TESTE     (KNN): {acc_knn_test:.4f}\n')
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_knn,
    display_labels=le.classes_,
    xticks_rotation='vertical',
    ax=ax, colorbar=False
)
ax.set_title('Matriz de Confusão – KNN (Teste)')
plt.tight_layout()
plt.savefig('../reports/figures/nb01_knn_matriz_confusao.png', dpi=120)
plt.show()


## 12. Comparação dos Algoritmos

In [ ]:
resumo = pd.DataFrame({
    'Melhores params'  : [str(gs_dt.best_params_), str(gs_knn.best_params_)],
    'Acurácia VAL'     : [acc_dt_val,  acc_knn_val],
    'Acurácia TESTE'   : [acc_dt_test, acc_knn_test],
}, index=['Árvore de Decisão', 'KNN'])
print(resumo.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(2)
width = 0.35
labels = ['Árvore de Decisão', 'KNN']
val_scores  = [acc_dt_val,  acc_knn_val]
test_scores = [acc_dt_test, acc_knn_test]

b1 = ax.bar(x - width/2, val_scores,  width, label='Validação', color='steelblue', alpha=0.7)
b2 = ax.bar(x + width/2, test_scores, width, label='Teste',     color='coral')

for bar, val in list(zip(b1, val_scores)) + list(zip(b2, test_scores)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Acurácia')
ax.set_ylim(0, 1.15)
ax.set_title('Comparação: Validação vs Teste')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/nb01_comparacao_dt_knn.png', dpi=120)
plt.show()


In [ ]:
f1_dt  = f1_score(y_test, y_pred_dt,  average=None)
f1_knn = f1_score(y_test, y_pred_knn, average=None)

x_pos = np.arange(len(le.classes_))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x_pos - width/2, f1_dt,  width, label='Árvore de Decisão', color='steelblue')
ax.bar(x_pos + width/2, f1_knn, width, label='KNN',               color='coral')
ax.set_xticks(x_pos)
ax.set_xticklabels(le.classes_, rotation=20, ha='right')
ax.set_ylabel('F1-score')
ax.set_title('F1-score por Classe (conjunto de teste)')
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig('../reports/figures/nb01_f1_por_classe.png', dpi=120)
plt.show()


## 13. Conclusão

### Resumo do processo de construção do dataset

| Etapa | Ação | Justificativa |
|---|---|---|
| 1 | Download do dataset UCI | Ponto de partida com 958 instâncias reais |
| 2 | Identificação de problemas | Apenas 2 classes; somente estados finais de partida |
| 3 | Reclassificação | `positive`→X vence; `negative`→O vence/Empate via lógica do jogo |
| 4 | Geração sintética | Simulação de partidas para "Tem jogo" e "Possibilidade de Fim de Jogo" |
| 5 | Deduplicação | Garantia de unicidade de cada estado no dataset |
| 6 | Balanceamento | ~200 amostras por classe para evitar viés nos classificadores |
| 7 | Codificação | Valores categóricos {b,o,x} → {0,1,2} |
| 8 | Divisão 60/20/20 | Treino / Validação / Teste — mesmos conjuntos para ambos os algoritmos |

### Divisão do dataset

| Conjunto | Arquivo | Uso |
|---|---|---|
| Treino (60%) | `treino.csv` | Ajuste dos pesos/parâmetros internos do modelo |
| Validação (20%) | `validacao.csv` | Seleção de hiperparâmetros (via GridSearchCV + PredefinedSplit) |
| Teste (20%) | `teste.csv` | Avaliação final imparcial — usado somente ao final |

### Limitação: classe Empate

Existem matematicamente apenas **16 posições de empate únicas** no jogo da velha quando X joga
primeiro. Isso é uma propriedade do jogo, não do dataset. Essa escassez impacta as métricas da classe.

### Resultados

In [ ]:
vencedor_nome = 'Árvore de Decisão' if acc_dt_test >= acc_knn_test else 'KNN'
vencedor_acc  = max(acc_dt_test, acc_knn_test)

print('=' * 60)
print('RESULTADOS FINAIS')
print('=' * 60)
print(f'Árvore de Decisão  → Val: {acc_dt_val:.4f}  |  Teste: {acc_dt_test:.4f}')
print(f'KNN                → Val: {acc_knn_val:.4f}  |  Teste: {acc_knn_test:.4f}')
print(f'\nMelhor algoritmo: {vencedor_nome} (acurácia teste = {vencedor_acc:.4f})')
print('=' * 60)
